# Chapter 0 — Prerequisites · Homework (self-test)

**Goal:** check that the foundations are solid before the vendor paths: LLM behavior and cost, REST and retries, JSON and JSON Schema, async Python, and testing output that varies.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 60–90 minutes
- **Estimated API cost:** $0. Everything here is offline; no API key is needed.
- **Prerequisites:** README read, practice notebook done

**How to use it**

1. Attempt everything **without peeking**. Solutions are at the very bottom, inside collapsed blocks.
2. **Part A:** 10 scenario questions on LLM fundamentals; put your letters in the `answers` dict and run the grader.
3. **Part B:** 7 auto-checked coding exercises (REST verbs and status codes, backoff, JSON parsing, JSON Schema, Pydantic validation, async, testing output that varies).
4. **Part C:** one architecture scenario; compare your answer with the rubric.
5. **Part D:** tick the readiness checklist (not scored).
6. **Scorecard:** overall score = (correct quiz answers + exercises passed) / 17. The pass mark is **72%**, which mirrors the Claude exam's 720/1000 and is used on both paths.

In [ ]:
import asyncio
import concurrent.futures
import hashlib
import json
import random
import re
import time
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

RESULTS: dict[str, bool] = {}

### Shared helpers (given)

`validate()` is a tiny JSON Schema validator for the subset used in this repo (`type`, `properties`, `required`, `enum`, `additionalProperties`). In real code use the `jsonschema` package or Pydantic. `run_sync()` runs a coroutine to completion from normal code, even inside Jupyter's already-running event loop.

In [ ]:
_TYPES = {"object": dict, "array": list, "string": str, "boolean": bool, "null": type(None)}


def _is_type(value, name: str) -> bool:
    if name in ("number", "integer"):
        ok = isinstance(value, (int, float)) and not isinstance(value, bool)
        return ok and (name == "number" or float(value).is_integer())
    return isinstance(value, _TYPES[name])


def validate(instance, schema: dict, path: str = "$") -> list[str]:
    """Return a list of error strings (empty list = valid)."""
    errors = []
    types = schema.get("type")
    if types is not None:
        names = types if isinstance(types, list) else [types]
        if not any(_is_type(instance, n) for n in names):
            return [f"{path}: expected {types}, got {type(instance).__name__}"]
    if "enum" in schema and instance not in schema["enum"]:
        errors.append(f"{path}: {instance!r} not in {schema['enum']}")
    if isinstance(instance, dict):
        props = schema.get("properties", {})
        for key in schema.get("required", []):
            if key not in instance:
                errors.append(f"{path}: missing required {key!r}")
        for key, value in instance.items():
            if key in props:
                errors += validate(value, props[key], f"{path}.{key}")
            elif schema.get("additionalProperties") is False:
                errors.append(f"{path}: unexpected field {key!r}")
    return errors


def run_sync(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)                      # plain Python: no loop running
    with concurrent.futures.ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()  # Jupyter: run in a fresh loop on a worker thread


print(validate({"a": 1}, {"type": "object", "required": ["a", "b"]}))

## Part A — Quiz

Choose one letter per question. Each question has exactly one best answer.

**Q1.** A team measured that its typical prompt is about 4,000 tokens on Claude Haiku 4.5 and is sizing a migration to Claude Sonnet 5.5 (budget and context limits) with that number. What should they do?

- A) Reuse the number: all Claude models share one tokenizer
- B) Re-count the real prompts with `messages.count_tokens` and the Sonnet 5.5 model ID: newer Claude models use a tokenizer that produces roughly 30% more tokens for the same text
- C) Use characters divided by 4, which is the same for every model
- D) Count with OpenAI's `responses.input_tokens.count`, which is exact for any vendor

**Q2.** Users of a support chatbot send short messages, yet the cost of each turn keeps rising as a conversation goes on. What is the most likely reason?

- A) Vendors raise the per-token price as a session gets longer
- B) Output tokens from earlier turns are billed again as output on every turn
- C) The vendor stores the conversation and charges storage per turn
- D) The API is stateless: the application resends the whole history on every turn, so input tokens grow each turn

**Q3.** A developer ports an old tutorial that passes `temperature=0` to make Claude Sonnet 5.5 "deterministic". With the current Anthropic Python SDK the call fails before reaching the API. What is the right fix?

- A) Pass `temperature` through `extra_body` so the SDK cannot block it
- B) Replace it with `top_k=1`, which is still supported
- C) Remove the sampling parameters, steer with prompting, `effort` and structured outputs, and make the output checkable (schemas, validators, tests) instead of relying on identical output
- D) Switch to streaming, which is deterministic

**Q4.** A parser reads `msg.content[0].text`. On Claude Sonnet 5.5 it sometimes fails with `AttributeError` even though `stop_reason` is `end_turn`. What is the cause and the fix?

- A) The first block can be a `thinking` block; select blocks by `type == "text"`
- B) `end_turn` responses have an empty `content` list; check `stop_reason` first
- C) The SDK returns plain dicts on adaptive-thinking models; use `msg.content[0]["text"]`
- D) `effort: "low"` disables text output; raise the effort

**Q5.** A CI test asserts that a ticket classifier replies exactly `"Category: billing"`. It passes locally and fails at random in CI. What is the best fix?

- A) Set `temperature=0` so the output never changes
- B) Re-run the failing test until it passes, and mark it flaky
- C) Delete the test: model output cannot be tested
- D) Constrain the output (a structured output with an enum of labels) and assert properties (it parses, the label is allowed and correct) over several runs on a small eval set, tracking a pass rate

**Q6.** An invoice-extraction pipeline returns plausible due dates for invoices that have no due date at all. Which change fixes this most directly?

- A) Raise `effort` to `max` so the model thinks harder
- B) Make `due_date` nullable in the schema (`["string", "null"]`), say in its description that `null` means "not on the invoice", and validate the output
- C) Move to a model with a larger context window
- D) Add "DO NOT HALLUCINATE" to the system prompt in capital letters

**Q7.** A team building retrieval-augmented generation (RAG) on Claude asks which Anthropic embeddings endpoint to call. What is the correct answer?

- A) Anthropic does not offer its own embedding model; use a separate provider (Anthropic's docs point to Voyage AI), OpenAI's embedding models, or a local model
- B) `POST /v1/embeddings` on `api.anthropic.com`
- C) `messages.count_tokens` returns the embedding vectors along with the count
- D) None: Claude cannot be used for RAG

**Q8.** Every incoming invoice must be extracted, validated against business rules, and posted to the ERP, always in that order. What is the best first design?

- A) An autonomous agent with every ERP tool that decides which steps to take
- B) A multi-agent system with a planner agent and three worker agents
- C) A workflow: a fixed sequence of model calls and code steps, with validation in code between them
- D) Fine-tune a model to output ERP API calls

**Q9.** A nightly job on the Claude API starts failing with HTTP 429. The response has no `retry-after` header and `error.details.error_code` is `enforced_spend_limit_reached`. The SDK's automatic retries fail too. What should the job do?

- A) Raise `max_retries` to 10 and add more jitter
- B) Rotate to another API key in the same organization
- C) Switch to `max_retries=0` and retry in a tight loop
- D) Treat it as non-transient: stop, alert a human, and resume after the spend cap is raised or the period resets

**Q10.** An agent's `issue_refund` tool call times out. The agent calls the tool again, and the customer is refunded twice. Which fix addresses the root cause?

- A) Give the side-effecting tool an idempotency key (for example derived from the conversation ID and the action) and make the backend ignore duplicates
- B) Disable retries in the SDK client
- C) Lower `effort` so the agent makes fewer tool calls
- D) Raise the timeout to 10 minutes

In [ ]:
answers = {"q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
           "q6": None, "q7": None, "q8": None, "q9": None, "q10": None}

In [ ]:
ANSWER_HASHES = {
    "q1": "96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365",
    "q2": "d14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b",
    "q3": "06217269d5c7af55ec1458d68670a9d1686eee2fa64726f5ac13566f0efad593",
    "q4": "0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0",
    "q5": "aa2a5c0e05888822a98d25f809f1adea4dcc57b38fe82026bca63c99ab417457",
    "q6": "621f4cab9e29d186b9dabeec79a433d2126c30a9e3f5712a1411fc67d73f1e40",
    "q7": "8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1",
    "q8": "89443923f22dc808ba069cfb289117719cbe7d8d36703c74314c6b70fba0371b",
    "q9": "e58517837d895fe6365a5801be9a66d9d1d39f1d863920a84cbed549cfec1954",
    "q10": "aeb5147fa5276b8759290ab8af9ed3aaa01d2c844cecc89ea5547b013e209ad3"
}


def grade_quiz(answers: dict) -> int:
    """Compare answers with hashed keys; unanswered counts as wrong."""
    correct = 0
    for qid, expected in ANSWER_HASHES.items():
        letter = str(answers.get(qid) or "").strip().upper()
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == expected
        correct += ok
        print(f"{qid:>4}: {letter or '-'} {'✓' if ok else '✗'}")
    print(f"Quiz score: {correct}/{len(ANSWER_HASHES)} ({correct / len(ANSWER_HASHES):.0%})")
    return correct


QUIZ_SCORE = grade_quiz(answers)

## Part B — Coding exercises

Each exercise has a task, a stub to complete, and a checker. Checkers never crash the notebook; they print ✓ or a hint.

### Exercise 1 — REST verbs and status codes: retry or not? ([§1.2](README.md#12-http--rest-and-json), [§1.8](README.md#18-software-engineering-baseline-checklist), [§3.3](README.md#33-retries-and-backoff))

First, fill `IDEMPOTENT_METHODS` with the verbs among `GET`, `POST`, `PUT`, `PATCH`, `DELETE` that are idempotent by definition (sending the same request twice leaves the server in the same state as sending it once).

Then write `retry_decision(status, body=None)` that returns:

| Response | Return |
|---|---|
| any 2xx | `"ok"` |
| 408, 409, any 5xx (including 529) | `"retry"` |
| 429 | `"retry"`, unless `body["error"]["details"]["error_code"] == "enforced_spend_limit_reached"`, then `"alert_human"` |
| any other status (400, 401, 402, 403, 404, 413, ...) | `"do_not_retry"` |

`body` may be `None` or lack `details`: don't crash.

In [ ]:
# TODO: which of GET, POST, PUT, PATCH, DELETE are idempotent by definition?
IDEMPOTENT_METHODS: set[str] = set()


def retry_decision(status: int, body: dict | None = None) -> str:
    """Return "ok", "retry", "do_not_retry" or "alert_human" for an HTTP response."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> bool:
    try:
        spend_cap = {"type": "error", "error": {"type": "rate_limit_error", "message": "cap",
                                                "details": {"error_code": "enforced_spend_limit_reached"}}}
        plain_429 = {"type": "error", "error": {"type": "rate_limit_error", "message": "slow down"}}
        cases = [(200, None, "ok"), (400, None, "do_not_retry"), (401, None, "do_not_retry"),
                 (402, None, "do_not_retry"), (403, None, "do_not_retry"), (404, None, "do_not_retry"),
                 (413, None, "do_not_retry"), (408, None, "retry"), (409, None, "retry"),
                 (429, plain_429, "retry"), (429, None, "retry"), (429, spend_cap, "alert_human"),
                 (500, None, "retry"), (503, None, "retry"), (504, None, "retry"), (529, None, "retry")]
        for status, body, expected in cases:
            got = retry_decision(status, body)
            assert got == expected, f"status {status}{' (spend cap)' if body is spend_cap else ''}: expected {expected!r}, got {got!r}"
        verbs = {"GET", "POST", "PUT", "PATCH", "DELETE"}
        chosen = {m.upper() for m in IDEMPOTENT_METHODS} & verbs
        assert chosen == {"GET", "PUT", "DELETE"}, (
            f"IDEMPOTENT_METHODS: got {sorted(chosen)}. Repeating the same GET, PUT or DELETE "
            "leaves the server in the same state; POST and PATCH may not")
        print("✓ Exercise 1 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 1: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 1: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 1: {type(e).__name__}: {e}")
    return False


RESULTS['ex1'] = check_ex1()

### Exercise 2 — Exponential backoff with jitter ([§3.3](README.md#33-retries-and-backoff))

Write `backoff_delay(attempt, retry_after=None, *, base=1.0, cap=30.0, rand=random.random)`:

- If `retry_after` is given (a number or a numeric string, in seconds), return it as a float, capped at `cap`.
- Otherwise use **full jitter**: `rand() * min(cap, base * 2 ** attempt)`. `attempt` starts at 0.

`rand` is injectable so the checker can make it deterministic.

In [ ]:
def backoff_delay(attempt: int, retry_after: float | str | None = None, *,
                  base: float = 1.0, cap: float = 30.0, rand=random.random) -> float:
    """Seconds to wait before retry number `attempt` (0-based)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> bool:
    try:
        top = lambda: 1.0          # noqa: E731  (largest possible jitter)
        expected = [1, 2, 4, 8, 16, 30, 30]
        got = [backoff_delay(a, rand=top) for a in range(7)]
        assert got == expected, f"with rand()=1.0 expected {expected} (doubling, capped at 30), got {got}"
        assert backoff_delay(3, rand=lambda: 0.5) == 4, "full jitter: delay = rand() * min(cap, base * 2**attempt)"
        assert backoff_delay(0, retry_after="7", rand=top) == 7.0, "a retry-after value (even a string) wins"
        assert backoff_delay(0, retry_after=120, rand=top) == 30, "cap retry-after at `cap` too"
        rng = random.Random(42)
        for a in range(10):
            d = backoff_delay(a, rand=rng.random)
            assert 0 <= d <= min(30, 2 ** a), f"attempt {a}: {d} outside [0, {min(30, 2 ** a)}]"
        delays = {backoff_delay(4, rand=rng.random) for _ in range(20)}
        assert len(delays) > 1, "jitter: delays for the same attempt should differ between calls"
        print("✓ Exercise 2 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 2: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 2: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 2: {type(e).__name__}: {e}")
    return False


RESULTS['ex2'] = check_ex2()

### Exercise 3 — Parse JSON out of model text ([§1.8](README.md#18-software-engineering-baseline-checklist))

Without structured outputs, models often wrap JSON in a ```` ```json ```` fence or surround it with prose. Write `parse_model_json(text)` that returns the first JSON **object** (a `dict`) it finds, or `None` if there is no valid object. It must never raise. Hint: `json.JSONDecoder().raw_decode` parses a JSON value from the start of a string and ignores what follows.

In [ ]:
def parse_model_json(text: str) -> dict | None:
    """Extract one JSON object from model text; return None if there is none."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> bool:
    try:
        assert parse_model_json('{"total": 42}') == {"total": 42}, "plain JSON"
        assert parse_model_json('```json\n{"total": 42, "currency": "EUR"}\n```') == {"total": 42, "currency": "EUR"}, "fenced ```json block"
        assert parse_model_json('Here is the result:\n{"a": {"b": [1, 2]}}\nHope this helps!') == {"a": {"b": [1, 2]}}, \
            "JSON object surrounded by prose (nested braces)"
        assert parse_model_json('{"total": 42,}') is None, "invalid JSON (trailing comma) -> None, not an exception"
        assert parse_model_json("I could not find an invoice.") is None, "no JSON at all -> None"
        assert parse_model_json("[1, 2, 3]") is None, "a top-level array is not the object we asked for -> None"
        assert parse_model_json("") is None, "empty text -> None"
        print("✓ Exercise 3 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 3: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 3: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 3: {type(e).__name__}: {e}")
    return False


RESULTS['ex3'] = check_ex3()

### Exercise 4 — Write a JSON Schema by hand ([§1.3](README.md#13-json-schema-basics))

Without looking at the README example, write `INVOICE_SCHEMA` for an object with:

- `invoice_id`: string · `total`: number · `currency`: one of `USD`, `EUR`, `GBP` · `due_date`: a string **or null**
- all four fields required, no other fields allowed

The checker validates seven instances with the small `validate()` helper above.

In [ ]:
INVOICE_SCHEMA: dict = {
    # TODO: write the JSON Schema by hand
}

In [ ]:
def check_ex4() -> bool:
    try:
        s = INVOICE_SCHEMA
        assert s.get("type") == "object", "top-level type must be object"
        assert set(s.get("required", [])) == {"invoice_id", "total", "currency", "due_date"}, \
            "all four fields are required (a nullable field is still required)"
        assert s.get("additionalProperties") is False, "reject unexpected fields with additionalProperties: false"
        good = {"invoice_id": "INV-7", "total": 120.5, "currency": "EUR", "due_date": "2026-11-01"}
        cases = [
            (good, True, "a complete invoice"),
            ({**good, "due_date": None}, True, "due_date null (not on the invoice)"),
            ({k: v for k, v in good.items() if k != "due_date"}, False, "due_date missing"),
            ({**good, "currency": "JPY"}, False, "currency outside the enum"),
            ({**good, "total": "120.5"}, False, "total as a string"),
            ({**good, "invoice_id": 7}, False, "invoice_id as a number"),
            ({**good, "notes": "x"}, False, "an extra field"),
        ]
        for instance, ok, label in cases:
            errors = validate(instance, s)
            assert (not errors) == ok, f"{label}: expected {'valid' if ok else 'invalid'}, validator said {errors or 'valid'}"
        print("✓ Exercise 4 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 4: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 4: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 4: {type(e).__name__}: {e}")
    return False


RESULTS['ex4'] = check_ex4()

### Exercise 5 — The same contract in Pydantic, with feedback for a retry ([§1.3](README.md#13-json-schema-basics), [§2.6](README.md#26-hallucination))

1. Define `class Invoice(BaseModel)` with the same four fields: `currency` as a `Literal`, `due_date` required **but** nullable (no default), and extra fields forbidden.
2. Write `validate_invoice(raw)` that parses a JSON string and returns `(invoice, None)` or `(None, error_message)`. The message is what you would send back to the model on a retry, so it should name the failing field.

In [ ]:
class Invoice(BaseModel):
    # TODO: fields and config
    pass


def validate_invoice(raw: str) -> tuple[Invoice | None, str | None]:
    """Return (invoice, None) if valid, else (None, error message to send back to the model)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> bool:
    try:
        good = '{"invoice_id": "INV-7", "total": 120.5, "currency": "EUR", "due_date": null}'
        inv, err = validate_invoice(good)
        assert err is None and isinstance(inv, Invoice), f"a valid invoice should parse, got error: {err}"
        assert inv.due_date is None and inv.total == 120.5, "fields should hold the parsed values"
        for raw, label in [
            ('{"invoice_id": "INV-7", "total": 120.5, "currency": "JPY", "due_date": null}', "currency outside USD/EUR/GBP"),
            ('{"invoice_id": "INV-7", "total": 120.5, "currency": "EUR"}', "due_date missing (required, even though it may be null)"),
            ('{"invoice_id": "INV-7", "total": 120.5, "currency": "EUR", "due_date": null, "notes": "x"}', "an extra field"),
            ('{"invoice_id": "INV-7", "total": 120.5,}', "invalid JSON"),
        ]:
            inv, err = validate_invoice(raw)
            assert inv is None and isinstance(err, str) and err, f"{label}: expected (None, error message), got ({inv!r}, {err!r})"
        _, err = validate_invoice('{"invoice_id": "INV-7", "total": 1, "currency": "JPY", "due_date": null}')
        assert "currency" in err, "the error message should name the failing field, so the model can fix it on retry"
        schema = Invoice.model_json_schema()
        assert set(schema.get("required", [])) == {"invoice_id", "total", "currency", "due_date"}, \
            f"generated schema should require all four fields, got {schema.get('required')}"
        assert schema.get("additionalProperties") is False, "forbid extra fields (model_config extra='forbid')"
        print("✓ Exercise 5 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 5: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 5: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 5: {type(e).__name__}: {e}")
    return False


RESULTS['ex5'] = check_ex5()

### Exercise 6 — Concurrent calls with a limit ([§1.1](README.md#11-python-311-typing-asyncawait-venv))

Write `async def run_all(prompts, call, max_concurrency=3)`:

- run `await call(prompt)` for every prompt **concurrently**, but never more than `max_concurrency` at once (rate limits!)
- return the results **in input order**
- if a call raises, put `f"ERROR: {type(e).__name__}"` in its slot instead of failing the whole batch

The checker runs it with `run_sync(...)` (given), which works both inside Jupyter and in a plain script.

In [ ]:
async def run_all(prompts: list[str], call, max_concurrency: int = 3) -> list[str]:
    """Run `await call(prompt)` for every prompt concurrently (at most `max_concurrency` at once)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> bool:
    try:
        state = {"now": 0, "peak": 0}

        async def fake_call(prompt: str) -> str:
            state["now"] += 1
            state["peak"] = max(state["peak"], state["now"])
            await asyncio.sleep(0.05)
            state["now"] -= 1
            if prompt == "boom":
                raise ValueError("simulated failure")
            return prompt.upper()

        prompts = [f"p{i}" for i in range(9)]
        t0 = time.perf_counter()
        out = run_sync(run_all(prompts, fake_call, max_concurrency=3))
        elapsed = time.perf_counter() - t0
        assert out == [p.upper() for p in prompts], f"results must be in input order, got {out}"
        assert state["peak"] == 3, f"at most 3 calls at once (and use all 3): peak was {state['peak']}"
        assert elapsed < 0.3, f"took {elapsed:.2f}s: calls should overlap (sequential would take 0.45s)"
        state["peak"] = 0
        out = run_sync(run_all(["a", "boom", "c"], fake_call, max_concurrency=5))
        assert out == ["A", "ERROR: ValueError", "C"], f"one failure must not sink the batch, got {out}"
        print("✓ Exercise 6 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 6: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 6: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 6: {type(e).__name__}: {e}")
    return False


RESULTS['ex6'] = check_ex6()

### Exercise 7 — Test output that varies ([§2.9](README.md#29-next-token-generation))

A ticket classifier must reply with a JSON object such as `{"label": "billing"}`, using one of `LABELS`. The model samples one token at a time, so the same ticket gets slightly different replies on different runs. An exact-match assertion on one string would be flaky. Test properties and a pass rate instead:

1. Write `is_acceptable(output, expected)`: `True` only if `output` parses as a JSON **object** whose only key is `"label"`, whose value is in `LABELS` and equals `expected`. It must never raise.
2. Write `pass_rate(outputs, expected)`: the share of acceptable outputs as a float from 0.0 to 1.0, and `0.0` for an empty list.

The checker runs your functions on 200 simulated replies to one ticket. In a real eval you run each case several times over a small labeled set, track the pass rate, and block a prompt or model change that lowers it.

In [ ]:
LABELS = {"billing", "shipping", "technical", "other"}


def is_acceptable(output: str, expected: str) -> bool:
    """True if `output` is a JSON object whose only key is "label", with an allowed label equal to `expected`."""
    # TODO
    raise NotImplementedError


def pass_rate(outputs: list[str], expected: str) -> float:
    """Share of acceptable outputs (0.0 to 1.0); 0.0 for an empty list."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex7() -> bool:
    try:
        cases = [
            ('{"label": "billing"}', "billing", True, "the ideal reply"),
            ('{"label":"billing"}', "billing", True, "only whitespace differs: parse and compare values, not strings"),
            ('{\n  "label": "billing"\n}', "billing", True, "pretty-printed JSON is the same value"),
            ("Category: billing", "billing", False, "prose instead of JSON"),
            ('{"label": "Billing"}', "billing", False, "labels are case-sensitive"),
            ('{"label": "shipping"}', "billing", False, "valid format, wrong label"),
            ('{"label": "refunds"}', "refunds", False, "a label outside LABELS is never acceptable"),
            ('{"label": "billing", "confidence": 0.9}', "billing", False, "an extra key breaks the contract"),
            ('{"label": "bill', "billing", False, "truncated JSON (the max_tokens case)"),
            ('["billing"]', "billing", False, "a JSON array, not an object"),
            ("", "billing", False, "empty output"),
        ]
        for output, expected, ok, why in cases:
            got = is_acceptable(output, expected)
            assert got is ok, f"is_acceptable({output!r}, {expected!r}) should be {ok} ({why}), got {got!r}"
        assert pass_rate([], "billing") == 0.0, "pass_rate([]) must return 0.0, not raise ZeroDivisionError"
        variants = ['{"label": "billing"}', '{"label":"billing"}', "Category: billing", '{"label": "shipping"}']
        outputs = random.Random(3).choices(variants, weights=[80, 8, 7, 5], k=200)   # a "sampled" model
        expected_rate = sum(o in variants[:2] for o in outputs) / len(outputs)
        got = pass_rate(outputs, "billing")
        assert isinstance(got, float) and abs(got - expected_rate) < 1e-9, f"pass_rate: expected {expected_rate:.3f}, got {got!r}"
        print("✓ Exercise 7 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 7: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 7: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 7: {type(e).__name__}: {e}")
    return False


RESULTS['ex7'] = check_ex7()

## Part C — Architecture scenario

**Scenario.** A 40-person support team wants two things from an LLM:

1. A **nightly job** that summarizes the day's ~5,000 closed tickets (about 1,500 input and 300 output tokens each) and writes each summary back to the ticket system.
2. A **daytime assistant** that answers agents' questions about a ticket in real time.

The CFO wants a monthly cost estimate before approving, security wants to know how keys are handled, and the team lead asks how you will know the summaries are good.

**Your task.** Describe: (1) the cost estimate for the nightly job on two candidate models, with the arithmetic, and how the Batch API changes it; (2) sync vs. batch for each workload; (3) failure handling: which errors you retry and how, and what you do on a spend-cap error; (4) how a re-run of the nightly job avoids writing duplicate summaries; (5) what you log per call; (6) key handling and spend guardrails; (7) how you test summary quality given that outputs vary. See [§3.1](README.md#31-the-latency--cost--reliability-triangle)–[§3.5](README.md#35-security-and-secrets) and [Step 7](README.md#step-7--put-cost-guardrails-in-place-before-the-first-call).

_Write your answer in the markdown cell below._

**My answer:**

1. Cost estimate:
2. Sync vs. batch:
3. Failure handling:
4. Duplicate-safe re-runs:
5. Logging:
6. Keys and guardrails:
7. Quality testing:

<details><summary>Rubric: what a strong answer contains</summary>

- **Cost, with arithmetic.** Per ticket on Sonnet 5.5: 1,500 × $2/M + 300 × $10/M = $0.003 + $0.003 = **$0.006**; × 5,000 = **$30 per night**, about **$900 per 30-day month**; with the Batch API (50% off) **$15 per night / about $450 per month**. On `gpt-6-luna`: 1,500 × $0.10/M + 300 × $0.50/M = $0.0003 per ticket, **$1.50 per night** (about $0.75 batched). The model is chosen by an eval on real tickets, not by price alone; a shared, cached instruction prefix lowers the input part further. Haiku 4.5 is cheaper than Sonnet, but its earliest retirement date is days away, so a plan built on it needs a migration path.
- **Sync vs. batch.** The nightly job can wait, so it goes to the Batch API (results within 24 hours, usually much sooner). The assistant is interactive, so it is synchronous, ideally streamed for perceived latency, with low `effort` for simple questions.
- **Failures.** Rely on the SDK's retries (408/409/429/5xx/529, exponential backoff, `retry-after`), tune `max_retries` and `timeout` per client, cap total attempts and elapsed time, and log each retry. 400/401/403/404/413 are fixed, not retried. A spend-cap 429 (`enforced_spend_limit_reached`, no `retry-after`) stops the job and alerts a human. For batches, re-submit only the failed or expired items.
- **Idempotency.** Writing a summary is a side effect. Use the ticket ID (plus a summary version) as the idempotency key or `custom_id`, and upsert instead of insert, so a re-run or a duplicate result overwrites rather than duplicates.
- **Logging.** Per call: timestamp, provider, model, request ID, latency, input/output/cache tokens, `stop_reason` or status, estimated cost, attempt number, ticket ID; redact personal data in prompts and outputs before storing them; daily cost totals with an alert.
- **Keys and guardrails.** Keys in a secret manager or environment variables, never in code or notebooks; a separate key and Workspace/Project per environment (dev, CI, prod); spend limits and alerts on both platforms; least-privilege write access for the job.
- **Quality.** A labeled sample of tickets with reference facts; property checks (length, required fields, no invented ticket facts) plus a rubric-based review; track a pass rate over several runs and re-run it on every prompt or model change; sample production outputs for human review.
- **Red flags:** "set temperature to 0 so it's deterministic"; no cost arithmetic; synchronous calls for the nightly job without justification; retrying every error forever; keys pasted in code; inserting summaries without a duplicate check.
</details>

## Part D — Readiness checklist (not scored)

Set each item to `True` only if it is true **today**. The scorecard lists what is still open, with the README section to revisit.

In [ ]:
READY = {
    "venv active and `pip install -r requirements.txt` succeeded (practice 0.1 all OK)": False,
    "both keys in .env, .env git-ignored, never printed (practice 0.2)": False,
    "spend limits set on the Claude Console and the OpenAI dashboard": False,
    "live smoke test returned a reply from both Claude and OpenAI (practice 0.6, RUN_LIVE=True)": False,
    "I can estimate monthly cost from volume, token counts and a price table": False,
    "I can open a pull request from a branch and know what to do if a secret is committed": False,
    "I can read a short TypeScript SDK sample and say what request it sends": False,
}
WHERE = ["Step 3", "Step 6 and §3.5", "Step 7", "Step 9", "Step 7", "§1.4 and §1.8", "§1.1"]

## Scorecard

In [ ]:
quiz = globals().get("QUIZ_SCORE", 0)
exercises = ["ex1", "ex2", "ex3", "ex4", "ex5", "ex6", "ex7"]
passed = sum(bool(globals().get("RESULTS", {}).get(ex)) for ex in exercises)
overall = (quiz + passed) / 17
print(f"Quiz: {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises passed: {passed}/{len(exercises)}")
print(f"Overall: {overall:.0%} -> {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")
ready = globals().get("READY", {})
open_items = [(item, where) for (item, done), where in zip(ready.items(), WHERE) if not done]
print(f"\nReadiness: {len(ready) - len(open_items)}/{len(ready)} ticked")
for item, where in open_items:
    print(f"  [ ] {item}  -> README {where}")

**Next:** start a vendor path. [Claude path, Chapter 1 — Claude API Fundamentals](../anthropic-claude/01-claude-api-fundamentals/README.md) · [OpenAI path, Chapter 1 — Responses API Fundamentals](../openai-codex/01-responses-api-fundamentals/README.md)

---

## Solutions (spoilers)

### Quiz answers

<details><summary>Q1</summary>

**B.** Claude 4.7 and later models (including Sonnet 5.5) use a newer tokenizer that yields roughly 30% more tokens than earlier models such as Haiku 4.5. Counts are per vendor and per model generation, so count with the target model. C is a rough rule of thumb, and D counts with a different vendor's tokenizer.
</details>

<details><summary>Q2</summary>

**D.** The model remembers nothing between calls. Each request carries the full history as input, so input tokens (and cost and latency) grow with conversation length unless you trim, summarize or cache. Prices per token do not change with session length (A), earlier output is resent as *input*, not billed again as output (B), and C is invented.
</details>

<details><summary>Q3</summary>

**C.** Current Claude models reject non-default `temperature`, `top_p` and `top_k` (400), and SDK v1.x no longer accepts them (`TypeError`). A and B still send unsupported sampling parameters. Even `temperature=0` never guaranteed determinism, so the design must make outputs checkable.
</details>

<details><summary>Q4</summary>

**A.** With adaptive thinking, responses can contain `thinking` blocks before the `text` block. Always select output by block type. B, C and D describe behaviors the API does not have.
</details>

<details><summary>Q5</summary>

**D.** Text is generated one sampled token at a time, so exact strings vary. Test properties and pass rates, and constrain the format. A is not possible on current Claude models and never guaranteed determinism anyway; B hides regressions; C gives up on quality control.
</details>

<details><summary>Q6</summary>

**B.** A required, non-nullable field forces the model to produce *something*. A nullable type gives it a legitimate way to say "not present", and validation catches the rest. More thinking (A) or context (C) does not remove the pressure to fill the field, and D is a weak instruction with no structural change.
</details>

<details><summary>Q7</summary>

**A.** Anthropic has no first-party embedding model; its embeddings guide points to Voyage AI as one provider. OpenAI offers embeddings directly. B and C do not exist, and D is false: retrieval and generation are separate steps.
</details>

<details><summary>Q8</summary>

**C.** The steps are known in advance, so a workflow is simpler, cheaper, faster and more predictable. Use an agent only when the path cannot be predicted. A and B add cost and unpredictability without need; D does not address control flow.
</details>

<details><summary>Q9</summary>

**D.** A tier spend-cap 429 keeps failing until access resumes; retrying only adds load. Keys in the same organization share the cap (B). Detect the `error_code` and escalate.
</details>

<details><summary>Q10</summary>

**A.** A timeout does not tell you whether the first call succeeded, and agents or retry logic may resend. Idempotency keys make duplicates harmless. B and D only change how often the problem shows up (and the SDK does not run your tools anyway); C does not prevent a repeated call.
</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
IDEMPOTENT_METHODS: set[str] = {"GET", "PUT", "DELETE"}   # POST and PATCH are not


def retry_decision(status: int, body: dict | None = None) -> str:
    if 200 <= status < 300:
        return "ok"
    if status == 429:
        details = ((body or {}).get("error") or {}).get("details") or {}
        if details.get("error_code") == "enforced_spend_limit_reached":
            return "alert_human"
        return "retry"
    if status in (408, 409) or status >= 500:      # includes 529 overloaded
        return "retry"
    return "do_not_retry"                          # 400, 401, 402, 403, 404, 413, ...
```
</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def backoff_delay(attempt: int, retry_after: float | str | None = None, *,
                  base: float = 1.0, cap: float = 30.0, rand=random.random) -> float:
    if retry_after is not None:
        return min(float(retry_after), cap)        # the server knows best, within our cap
    return rand() * min(cap, base * 2 ** attempt)  # exponential growth, full jitter
```
</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def parse_model_json(text: str) -> dict | None:
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    candidate = fenced.group(1) if fenced else text
    start = candidate.find("{")
    if start == -1:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(candidate[start:])
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None
```
</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
INVOICE_SCHEMA: dict = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string", "description": "Invoice number exactly as printed"},
        "total": {"type": "number"},
        "currency": {"type": "string", "enum": ["USD", "EUR", "GBP"]},
        "due_date": {"type": ["string", "null"], "description": "ISO 8601 date, or null if absent"},
    },
    "required": ["invoice_id", "total", "currency", "due_date"],
    "additionalProperties": False,
}
```
</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
class Invoice(BaseModel):
    model_config = ConfigDict(extra="forbid")

    invoice_id: str
    total: float
    currency: Literal["USD", "EUR", "GBP"]
    due_date: str | None = Field(description="ISO 8601 date, or null if absent")  # required but nullable


def validate_invoice(raw: str) -> tuple[Invoice | None, str | None]:
    try:
        return Invoice.model_validate_json(raw), None
    except ValidationError as e:
        return None, str(e)
```
</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
async def run_all(prompts: list[str], call, max_concurrency: int = 3) -> list[str]:
    sem = asyncio.Semaphore(max_concurrency)

    async def one(prompt: str) -> str:
        async with sem:
            try:
                return await call(prompt)
            except Exception as e:  # noqa: BLE001
                return f"ERROR: {type(e).__name__}"

    return list(await asyncio.gather(*(one(p) for p in prompts)))   # gather keeps input order
```
</details>

<details><summary>Exercise 7 — reference solution</summary>

```python
LABELS = {"billing", "shipping", "technical", "other"}


def is_acceptable(output: str, expected: str) -> bool:
    try:
        data = json.loads(output)              # compare parsed values, never raw strings
    except (json.JSONDecodeError, TypeError):
        return False                           # prose, truncated JSON, empty text
    return (isinstance(data, dict) and set(data) == {"label"}
            and data["label"] in LABELS and data["label"] == expected)


def pass_rate(outputs: list[str], expected: str) -> float:
    if not outputs:
        return 0.0
    return sum(is_acceptable(o, expected) for o in outputs) / len(outputs)
```
</details>